## 1. Setup

In [21]:
# Import libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json
import folium
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add prepol module to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import prepol.config as config
import prepol.helpers as helpers

pd.set_option('display.max_columns', None)

print("✓ Libraries loaded")
print(f"✓ Project root: {project_root}")

✓ Libraries loaded
✓ Project root: d:\BackupSupremo\Work\prepol-project


## 2. Load Model

In [22]:
# Load the latest trained model
model_dir = project_root / "model"
model_files = sorted(model_dir.glob(f"{config.MODEL_PREFIX}_*.joblib"))

if not model_files:
    raise FileNotFoundError(f"No model found in {model_dir}")

model_path = model_files[-1]
meta_path = model_path.with_name(model_path.stem.replace(config.MODEL_PREFIX, f"{config.MODEL_PREFIX}_meta") + ".json")

print(f"Loading model: {model_path.name}")

# Load model and metadata
rf_model = joblib.load(model_path)
with open(meta_path, 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_columns']

print(f"✓ Model loaded")
print(f"  Test R²: {metadata['metrics']['test']['r2']:.4f}")
print(f"  Features: {feature_cols}")

Loading model: rf_crime_model_20251119_1852.joblib
✓ Model loaded
  Test R²: 0.9110
  Features: ['time_period', 'y_norm', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_rol_3', 'y_rol_7', 'y_lag_1_vizinhos']


## 3. Load Panel Data

In [23]:
# Load panel data
panel_path = Path.cwd() / "prepol_out" / "PrePol_panel_export.parquet"

print(f"Loading panel data...")
df_panel = pd.read_parquet(panel_path)

# Convert Period to string for easier handling
if 'time_period' in df_panel.columns:
    if pd.api.types.is_period_dtype(df_panel['time_period']):
        df_panel['time_period_str'] = df_panel['time_period'].astype(str)
    else:
        df_panel['time_period_str'] = df_panel['time_period']

print(f"✓ Panel loaded: {df_panel.shape}")
print(f"  Date range: {df_panel['timestamp'].min()} to {df_panel['timestamp'].max()}")
print(f"  H3 cells: {df_panel['h3_cell'].nunique():,}")
print(f"  Time periods: {df_panel['time_period_str'].nunique()}")

df_panel.head()

Loading panel data...
✓ Panel loaded: (14468283, 12)
  Date range: 2013-01-01 00:00:00 to 2016-12-31 00:00:00
  H3 cells: 9,903
  Time periods: 1461


,h3_cell,time_period,timestamp,y,y_norm,y_lag_1,y_lag_2,y_lag_3,y_rol_3,y_rol_7,y_lag_1_vizinhos,time_period_str
0,89a81000003ffff,2013-01-01,2013-01-01,0,-0.163568,NaN,NaN,NaN,NaN,NaN,0,2013-01-01
1,89a81000003ffff,2013-01-02,2013-01-02,0,-0.163568,0.0,NaN,NaN,0.000000,0.00,0,2013-01-02
2,89a81000003ffff,2013-01-03,2013-01-03,0,-0.163568,0.0,0.0,NaN,0.000000,0.00,0,2013-01-03
3,89a81000003ffff,2013-01-04,2013-01-04,1,5.810758,0.0,0.0,0.0,0.000000,0.00,0,2013-01-04
4,89a81000003ffff,2013-01-05,2013-01-05,0,-0.163568,1.0,0.0,0.0,0.333333,0.25,0,2013-01-05


## 4. Select Target Week

In [24]:
# Select target date range (default: last week)
available_periods = sorted(df_panel['time_period_str'].unique())

# Configure date range here
# Options:
# 1. Last 7 days (default)
# 2. Custom range - modify start_idx and end_idx below

# Default: last 7 days
default_start_idx = len(available_periods) - 7
default_end_idx = len(available_periods) - 1

# For custom dates, find indexes:
# start_idx = available_periods.index('2016-12-01')  # Example custom start
# end_idx = available_periods.index('2016-12-31')     # Example custom end

start_idx = available_periods.index('2016-12-01') 
end_idx = available_periods.index('2016-12-08')

target_start = available_periods[start_idx]
target_end = available_periods[end_idx]
target_period = f"{target_start} to {target_end}"

print(f"🎯 Target date range: {target_period}")
print(f"   Total days: {end_idx - start_idx + 1}")
print(f"   Available range: {available_periods[0]} to {available_periods[-1]}")

# Filter data for target date range
df_target = df_panel[
    (df_panel['time_period_str'] >= target_start) & 
    (df_panel['time_period_str'] <= target_end)
].copy()

print(f"\n✓ Target range data: {len(df_target):,} records")
print(f"  Unique cells: {df_target['h3_cell'].nunique():,}")
print(f"  Actual crimes: {df_target['y'].sum():,}")
print(f"  Daily average: {df_target['y'].sum() / (end_idx - start_idx + 1):.1f} crimes/day")

🎯 Target date range: 2016-12-01 to 2016-12-08
   Total days: 8
   Available range: 2013-01-01 to 2016-12-31

✓ Target range data: 79,224 records
  Unique cells: 9,903
  Actual crimes: 9,405
  Daily average: 1175.6 crimes/day


## 5. Generate Predictions

In [25]:
# Prepare features for prediction
df_prep = df_target.copy()

# Convert Period to ordinal (required for model)
for col in df_prep.columns:
    if pd.api.types.is_period_dtype(df_prep[col]):
        df_prep[col] = df_prep[col].apply(lambda x: x.ordinal if pd.notna(x) else np.nan)

# Extract features and predict
X_target = df_prep[feature_cols].fillna(0)
y_pred = rf_model.predict(X_target)

# Add predictions to dataframe
df_target['y_pred'] = np.clip(y_pred, 0, None)  # No negative predictions

print(f"✓ Predictions generated")
print(f"  Total predicted crimes: {df_target['y_pred'].sum():.0f}")
print(f"  Cells with predictions > 0: {(df_target['y_pred'] > 0.5).sum():,}")

✓ Predictions generated
  Total predicted crimes: 9747
  Cells with predictions > 0: 7,612


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [26]:
# Cell 5A: Aggregate predictions by cell (temporal averaging)
# When predicting over multiple days, we need to aggregate predictions per cell
# to get a single probability estimate, avoiding count inflation from summing daily predictions

print("Aggregating predictions by cell...")

# LINEAR SCALING probability conversion function
def count_to_probability(count, max_count=5.0):
    """Convert crime count to probability using linear scaling.
    
    Simple linear transformation: probability = min(count / max_count, 1.0)
    
    This ensures:
    - 0 crimes → 0% probability
    - max_count crimes → 100% probability
    - Linear increase between 0 and max_count
    - Any count > max_count → capped at 100%
    
    Args:
        count: Predicted daily average crime count
        max_count: Count corresponding to 100% probability (default 5.0)
    
    Returns:
        Probability between 0 and 1
    """
    return min(count / max_count, 1.0)

# Add coordinates to df_target (cell centroid for each H3 cell)
print("Adding H3 cell coordinates...")
df_target['lat'] = df_target['h3_cell'].apply(lambda x: helpers.h3_to_geo(x)[0])
df_target['lon'] = df_target['h3_cell'].apply(lambda x: helpers.h3_to_geo(x)[1])

# Group by cell and aggregate
df_cell_agg = df_target.groupby('h3_cell').agg({
    'y': 'sum',           # Total actual crimes in period
    'y_pred': 'mean',     # Average predicted count per day
    'lat': 'first',       # Cell coordinates (same for all periods)
    'lon': 'first',
    'time_period_str': 'count'  # Number of days in aggregation
}).reset_index()

# Rename for clarity
df_cell_agg.rename(columns={'time_period_str': 'n_days'}, inplace=True)

# Calculate probability with LINEAR SCALING (max_count=5.0)
df_cell_agg['crime_probability'] = df_cell_agg['y_pred'].apply(
    lambda x: count_to_probability(x, max_count=1.0)
)
df_cell_agg['normalized_prob'] = df_cell_agg['crime_probability']

# Add total predicted crimes for comparison with actual
df_cell_agg['y_pred_total'] = df_cell_agg['y_pred'] * df_cell_agg['n_days']

# Distribution analysis
print(f"✓ Aggregated to {len(df_cell_agg):,} unique cells")
print(f"  Average days per cell: {df_cell_agg['n_days'].mean():.1f}")
print(f"  Total actual crimes: {df_cell_agg['y'].sum():,}")
print(f"  Mean predicted daily count: {df_cell_agg['y_pred'].mean():.3f}")
print(f"  Median predicted daily count: {df_cell_agg['y_pred'].median():.3f}")

print(f"\n📊 Probability Distribution (Linear Scaling):")
print(f"  • Range: {df_cell_agg['crime_probability'].min()*100:.1f}% to {df_cell_agg['crime_probability'].max()*100:.1f}%")
print(f"  • Mean: {df_cell_agg['crime_probability'].mean()*100:.1f}%")
print(f"  • Median: {df_cell_agg['crime_probability'].median()*100:.1f}%")
print(f"  • Cells with <20% probability: {(df_cell_agg['crime_probability'] < 0.2).sum():,}")
print(f"  • Cells with 20-50% probability: {((df_cell_agg['crime_probability'] >= 0.2) & (df_cell_agg['crime_probability'] < 0.5)).sum():,}")
print(f"  • Cells with 50-80% probability: {((df_cell_agg['crime_probability'] >= 0.5) & (df_cell_agg['crime_probability'] < 0.8)).sum():,}")
print(f"  • Cells with >80% probability: {(df_cell_agg['crime_probability'] >= 0.8).sum():,}")

print(f"\n🎯 Daily Average Predictions:")
print(f"  • Min: {df_cell_agg['y_pred'].min():.4f}")
print(f"  • 25th percentile: {df_cell_agg['y_pred'].quantile(0.25):.4f}")
print(f"  • Median: {df_cell_agg['y_pred'].quantile(0.5):.4f}")
print(f"  • 75th percentile: {df_cell_agg['y_pred'].quantile(0.75):.4f}")
print(f"  • Max: {df_cell_agg['y_pred'].max():.4f}")

df_cell_agg.head()


Aggregating predictions by cell...
Adding H3 cell coordinates...
✓ Aggregated to 9,903 unique cells
  Average days per cell: 8.0
  Total actual crimes: 9,405
  Mean predicted daily count: 0.123
  Median predicted daily count: 0.000

📊 Probability Distribution (Linear Scaling):
  • Range: 0.0% to 100.0%
  • Mean: 11.3%
  • Median: 0.0%
  • Cells with <20% probability: 7,982
  • Cells with 20-50% probability: 1,467
  • Cells with 50-80% probability: 276
  • Cells with >80% probability: 178

🎯 Daily Average Predictions:
  • Min: 0.0000
  • 25th percentile: 0.0000
  • Median: 0.0000
  • 75th percentile: 0.1388
  • Max: 4.8944


,h3_cell,y,y_pred,lat,lon,n_days,crime_probability,normalized_prob,y_pred_total
0,89a81000003ffff,0,0.000009,-23.709493,-46.633485,8,0.000009,0.000009,0.000068
1,89a8100000bffff,0,0.000010,-23.708105,-46.630383,8,0.000010,0.000010,0.000077
2,89a8100000fffff,0,0.000010,-23.706432,-46.633395,8,0.000010,0.000010,0.000077
3,89a81000013ffff,0,0.000008,-23.712554,-46.633575,8,0.000008,0.000008,0.000065
4,89a81000017ffff,0,0.000008,-23.710882,-46.636587,8,0.000008,0.000008,0.000062


## 6. Create Interactive Map

In [27]:
# Add coordinates to each cell (using aggregated data)
print("Preparing map data...")

# Use aggregated cell data from Cell 10
df_map_source = df_cell_agg.copy()

print(f"✓ Using aggregated cell data: {len(df_map_source):,} unique cells")
print(f"  Average predictions: {df_map_source['y_pred'].mean():.3f} crimes/day")
print(f"  Probability already calculated in Cell 10")
print(f"  Probability range: {df_map_source['crime_probability'].min()*100:.1f}% to {df_map_source['crime_probability'].max()*100:.1f}%")
print(f"  Cells with >50% probability: {(df_map_source['crime_probability'] > 0.5).sum():,}")

# Calculate map center from aggregated data
center_lat = df_map_source['lat'].median()
center_lon = df_map_source['lon'].median()

print(f"✓ Map center: ({center_lat:.4f}, {center_lon:.4f})")


Preparing map data...
✓ Using aggregated cell data: 9,903 unique cells
  Average predictions: 0.123 crimes/day
  Probability already calculated in Cell 10
  Probability range: 0.0% to 100.0%
  Cells with >50% probability: 454
✓ Map center: (-23.5697, -46.6486)


In [28]:
# Create map with predicted crime probabilities (OPTIMIZED)
print("Creating crime probability map...")
start_time = time.time()

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Add alternative tile layers
folium.TileLayer('CartoDB positron').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Color function (same as H3Discretization: light red -> dark red)
def get_red_gradient(normalized_value):
    """Return hex color from light red to dark red based on normalized value (0-1)."""
    # RGB: (255, 200, 200) -> (139, 0, 0)
    r = int(255 - (116 * normalized_value))
    g = int(200 - (200 * normalized_value))
    b = int(200 - (200 * normalized_value))
    return f'#{r:02x}{g:02x}{b:02x}'

# Filter cells with meaningful crime probability (> 10%)
df_map = df_cell_agg[df_cell_agg['crime_probability'] > 0.1].copy()
print(f"Drawing {len(df_map):,} cells with probability > 10%...")

# Calculate percentile thresholds for legend
p33 = df_map['crime_probability'].quantile(0.33)
p67 = df_map['crime_probability'].quantile(0.67)
min_prob = df_map['crime_probability'].min()
max_prob = df_map['crime_probability'].max()

# OPTIMIZATION: Build GeoJSON FeatureCollection
features = []

# Vectorize boundary and color computation
df_map['fill_color'] = df_map['normalized_prob'].apply(get_red_gradient)
df_map['boundary'] = df_map['h3_cell'].apply(lambda x: helpers.h3_to_boundary(x))

for idx, row in df_map.iterrows():
    # Convert boundary to GeoJSON format (lon, lat)
    coords = [[[lon, lat] for lat, lon in row['boundary']]]
    
    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'Polygon',
            'coordinates': coords
        },
        'properties': {
            'fillColor': row['fill_color'],
            'color': row['fill_color'],
            'weight': 1,
            'fillOpacity': 0.6,
            'popup': (
                f"<b>Crime Occurrence Probability</b><br>"
                f"<b>Cell:</b> {row['h3_cell'][:8]}...<br>"
                f"<b>Probability:</b> {row['crime_probability']*100:.1f}%<br>"
                f"<b>Period:</b> {target_period} ({row['n_days']} days)<br>"
                f"<hr style='margin:4px 0'>"
                f"<b>Daily Avg Prediction:</b> {row['y_pred']:.3f}/day<br>"
                f"<b>Total Predicted:</b> {row['y_pred_total']:.1f}<br>"
                f"<b>Total Actual:</b> {row['y']}<br>"
                f"<b>Prediction Error:</b> {abs(row['y_pred_total'] - row['y']):.1f}"
            ),
            'tooltip': f"Probability: {row['crime_probability']*100:.1f}%"
        }
    }
    features.append(feature)

# Create single GeoJSON layer
geojson_data = {
    'type': 'FeatureCollection',
    'features': features
}

folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        'fillColor': feature['properties']['fillColor'],
        'color': feature['properties']['fillColor'],
        'weight': feature['properties']['weight'],
        'fillOpacity': feature['properties']['fillOpacity']
    },
    popup=folium.GeoJsonPopup(fields=['popup'], labels=False),
    tooltip=folium.GeoJsonTooltip(fields=['tooltip'], labels=False)
).add_to(m)

# Enhanced legend with percentile-based thresholds
legend_html = f'''
<div style="position: fixed; 
     bottom: 10px; left: 10px; width: 280px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:13px;
     padding: 12px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3)">
     <p style="margin:0 0 8px 0"><b>Crime Occurrence Probability</b></p>
     
     <p style="margin:5px 0"><span style="background-color:#ffc8c8; padding:2px 10px">▮</span> 
        Low ({min_prob*100:.1f}% - {p33*100:.1f}%)</p>
     <p style="margin:5px 0"><span style="background-color:#ff6464; padding:2px 10px">▮</span> 
        Medium ({p33*100:.1f}% - {p67*100:.1f}%)</p>
     <p style="margin:5px 0"><span style="background-color:#8b0000; padding:2px 10px">▮</span> 
        High ({p67*100:.1f}% - {max_prob*100:.1f}%)</p>
     
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ddd">
     
     <p style="margin:5px 0; font-size:11px; color:#333"><b>Period:</b> {target_period}</p>
     <p style="margin:5px 0; font-size:11px; color:#333"><b>Date Range:</b> {target_start} to {target_end}</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Aggregation:</b> Daily (H3 Res 9)</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Thresholds:</b> 33rd/67th percentile</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Display Filter:</b> Probability > 10%</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)

elapsed = time.time() - start_time
print(f"✓ Map created in {elapsed:.2f} seconds!")
print(f"\nMap Statistics:")
print(f"  • Cells displayed: {len(df_map):,}")
print(f"  • Probability range: {min_prob*100:.1f}% to {max_prob*100:.1f}%")
print(f"  • Percentile thresholds:")
print(f"    - 33rd percentile (Low/Medium): {p33*100:.1f}%")
print(f"    - 67th percentile (Medium/High): {p67*100:.1f}%")
print(f"  • High probability (>{p67*100:.1f}%): {(df_map['crime_probability'] > p67).sum()}")
print(f"  • Medium probability ({p33*100:.1f}%-{p67*100:.1f}%): {((df_map['crime_probability'] >= p33) & (df_map['crime_probability'] <= p67)).sum()}")
print(f"  • Low probability (<{p33*100:.1f}%): {(df_map['crime_probability'] < p33).sum()}")

m

Creating crime probability map...
Drawing 4,310 cells with probability > 10%...
✓ Map created in 0.46 seconds!

Map Statistics:
  • Cells displayed: 4,310
  • Probability range: 12.6% to 100.0%
  • Percentile thresholds:
    - 33rd percentile (Low/Medium): 13.1%
    - 67th percentile (Medium/High): 25.9%
  • High probability (>25.9%): 1422
  • Medium probability (13.1%-25.9%): 1466
  • Low probability (<13.1%): 1422


## 7. Summary

In [29]:
# Production testing summary
print("="*60)
print(" "*15 + "PREPOL DEMO SUMMARY")
print("="*60)

print(f"\n📅 Target Period: {target_period}")
print(f"🗺️  Unique H3 Cells: {len(df_cell_agg):,}")
print(f"📆 Days in Period: {df_cell_agg['n_days'].iloc[0]}")

print(f"\n📊 Predictions:")
print(f"  • Total actual crimes: {df_cell_agg['y'].sum():,}")
print(f"  • Average daily prediction per cell: {df_cell_agg['y_pred'].mean():.3f}")
print(f"  • Cells with predictions > 1/day: {(df_cell_agg['y_pred'] > 1.0).sum():,}")

print(f"\n🎲 Crime Occurrence Probability:")
print(f"  • Mean probability: {df_cell_agg['crime_probability'].mean()*100:.1f}%")
print(f"  • Median probability: {df_cell_agg['crime_probability'].median()*100:.1f}%")
print(f"  • Cells with >50% chance: {(df_cell_agg['crime_probability'] > 0.5).sum():,}")
print(f"  • Cells with >70% chance: {(df_cell_agg['crime_probability'] > 0.7).sum():,}")

print(f"\n🎯 Top 5 Highest Risk Areas:")
high_risk = df_cell_agg.nlargest(5, 'crime_probability')[['h3_cell', 'crime_probability', 'y_pred', 'y', 'n_days']]
for idx, row in high_risk.iterrows():
    print(f"  • Cell {row['h3_cell'][:8]}... → {row['crime_probability']*100:.1f}% chance (Avg: {row['y_pred']:.2f}/day, Total: {row['y']})")

print(f"\n💡 Map Features:")
print(f"  • Colors show PROBABILITY of crime occurrence (0-100%)")
print(f"  • Light red = Low probability (<30%)")
print(f"  • Medium red = Medium probability (30-70%)")
print(f"  • Dark red = High probability (>70%)")
print(f"  • Click cells for details")
print(f"  • Toggle map layers in top-right corner")

print(f"\n✅ Demo Complete!")
print("="*60)

               PREPOL DEMO SUMMARY

📅 Target Period: 2016-12-01 to 2016-12-08
🗺️  Unique H3 Cells: 9,903
📆 Days in Period: 8

📊 Predictions:
  • Total actual crimes: 9,405
  • Average daily prediction per cell: 0.123
  • Cells with predictions > 1/day: 129

🎲 Crime Occurrence Probability:
  • Mean probability: 11.3%
  • Median probability: 0.0%
  • Cells with >50% chance: 454
  • Cells with >70% chance: 226

🎯 Top 5 Highest Risk Areas:
  • Cell 89a81000... → 100.0% chance (Avg: 1.03/day, Total: 14)
  • Cell 89a81000... → 100.0% chance (Avg: 1.05/day, Total: 11)
  • Cell 89a81001... → 100.0% chance (Avg: 1.08/day, Total: 8)
  • Cell 89a81002... → 100.0% chance (Avg: 1.38/day, Total: 10)
  • Cell 89a81005... → 100.0% chance (Avg: 1.05/day, Total: 19)

💡 Map Features:
  • Colors show PROBABILITY of crime occurrence (0-100%)
  • Light red = Low probability (<30%)
  • Medium red = Medium probability (30-70%)
  • Dark red = High probability (>70%)
  • Click cells for details
  • Toggle map l